In [2]:
!pip install datasets huggingface_hub pandas

^C


  Using cached dill-0.3.8-py3-none-any.whl.metadata (10 kB)
  Using cached xxhash-3.5.0-cp311-cp311-win_amd64.whl.metadata (13 kB)
  Using cached multiprocess-0.70.16-py311-none-any.whl.metadata (7.2 kB)
  Using cached PyYAML-6.0.2-cp311-cp311-win_amd64.whl.metadata (2.1 kB)
  Using cached frozenlist-1.5.0-cp311-cp311-win_amd64.whl.metadata (14 kB)
Using cached dill-0.3.8-py3-none-any.whl (116 kB)
Using cached multiprocess-0.70.16-py311-none-any.whl (143 kB)
   ---------------------------------------- 0.0/25.3 MB ? eta -:--:--
   -------- ------------------------------- 5.2/25.3 MB 24.4 MB/s eta 0:00:01
   --------------- ------------------------ 10.0/25.3 MB 23.8 MB/s eta 0:00:01
   ----------------------- ---------------- 14.7/25.3 MB 24.3 MB/s eta 0:00:01
   ------------------------------- -------- 19.9/25.3 MB 23.7 MB/s eta 0:00:01
   -------------------------------------- - 24.6/25.3 MB 24.0 MB/s eta 0:00:01
   ---------------------------------------- 25.3/25.3 MB 23.2 MB/s eta 0:

In [1]:
import os
import pandas as pd
import json

# Load Excel
reports_df = pd.read_excel("../../resources/listado-informes.xlsx")

# File to save the current state
progress_file = "prompt_iteration_progress.json"


## Generate prompts

In [2]:
def load_progress():
    if os.path.exists(progress_file):
        with open(progress_file, "r", encoding="utf-8") as file:
            progress = json.load(file)
        return progress.get("current_row", 0)
    return 0

def save_progress(current_row):
    with open(progress_file, "w", encoding="utf-8") as file:
        json.dump({"current_row": current_row}, file)


In [3]:
def load_combined_prompts(ident, title):
    combined_prompt = ""
    for prompt_number in range(1, 4):
        prompt_path = f"prompts/basic_question{prompt_number}.txt"
        with open(prompt_path, "r", encoding="utf-8") as file:
            prompt = file.read().replace("{{report_name}}", f"{ident} {title}")
            combined_prompt += prompt + "\n\n"
    return combined_prompt.strip()


### AI instance Generation

In [ ]:
current_row = load_progress()
total_rows = len(reports_df)

print(f"Starting from row: {current_row}")

for idx in range(current_row, total_rows):
    row = reports_df.iloc[idx]
    ident = row['ident']
    title = row['title']

    print(f"\n{'='*80}\nReport {idx + 1}/{total_rows} | Ident: {ident} | Title: {title}\n{'='*80}\n")

    combined_prompt = load_combined_prompts(ident, title)

    initial_context = f"Te voy a dar 3 prompts para que me generes tres instancias y metas las tres en un texto formato json. NO ME DES codigo python o algo asi. dame directamente el texto. \n\n"

    print(initial_context+combined_prompt)
    print("\n" + "-"*80)

    input("Press Enter to continue to the next report...")

    # Guarda el progreso inmediatamente después de mostrar cada informe
    save_progress(idx + 1)
    print(f"Progress saved at row {idx + 1}")

print("\n🎉 All reports processed!")


## Save examples

In [6]:
import json

# Cargar el archivo original
with open("instances_generated.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Agregar el campo "generated_by" a cada objeto
for item in data:
    item["generated_by"] = "gpt-4o"

# Guardar el nuevo archivo
with open("instances_generated_enriched.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("Archivo enriquecido guardado como data_enriched.json")


Archivo enriquecido guardado como data_enriched.json


In [ ]:
from datasets import Dataset
import pandas as pd

# Cargar el nuevo JSON enriquecido como DataFrame
df = pd.read_json("data_enriched.json")

# Convertir a Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Subir al Hub (sobrescribirá el dataset anterior, que ya estaba vacío)
dataset.push_to_hub("jdavit/colombian-conflict-SQA")

print("Nuevas instancias subidas exitosamente a jdavit/colombian-conflict-SQA")
